In [37]:
import random

# ==========================================
# 1. ENTIDAD DEL ENTORNO
# ==========================================
class Environment:
    def __init__(self, locations=['A', 'B','C','D'], initial_dirt=None, initial_agent_loc='A'):
        """Entorno configurable (Criterio 7)"""
        self.locations = locations
        # Si no se especifica, se genera suciedad aleatoria
        if initial_dirt is None:
            self.dirt_status = {loc: random.choice(['Limpio', 'Sucio']) for loc in locations}
        else:
            self.dirt_status = initial_dirt.copy()

        self.agent_location = initial_agent_loc
        self.performance_score = 0 # (Criterio 3)

    def get_percept(self):
        """Genera la percepción para el agente (Sensores)"""
        return (self.agent_location, self.dirt_status[self.agent_location])

    def execute_action(self, action):
        """Lógica de transición correcta y cálculo de rendimiento (Criterio 2 y 3)"""
        if action == 'Aspirar':
            if self.dirt_status[self.agent_location] == 'Sucio':
                self.dirt_status[self.agent_location] = 'Limpio'
                self.performance_score += 10  # Recompensa por limpiar

        elif action == 'Derecha':
            idx = self.locations.index(self.agent_location)
            if idx < len(self.locations) - 1:
                self.agent_location = self.locations[idx + 1]
            self.performance_score -= 1  # Penalización por moverse

        elif action == 'Izquierda':
            idx = self.locations.index(self.agent_location)
            if idx > 0:
                self.agent_location = self.locations[idx - 1]
            self.performance_score -= 1  # Penalización por moverse

        elif action == 'NoOp':
            pass # No penaliza ni recompensa

    def is_clean(self):
        """Verifica si todo el mundo está limpio"""
        return all(status == 'Limpio' for status in self.dirt_status.values())

# ==========================================
# 2. ENTIDADES DE LOS AGENTES
# ==========================================
class Agent:
    def act(self, percept):
        raise NotImplementedError("Debe implementarse en las subclases")

class RandomAgent(Agent):
    """Agente que elige acciones al azar (Criterio 8)"""
    def act(self, percept):
        return random.choice(['Izquierda', 'Derecha', 'Aspirar', 'NoOp'])

class SimpleReflexAgent(Agent):
    """Agente Reactivo Simple basado en reglas (Criterio 8)"""
    def act(self, percept):
        location, status = percept

        if status == 'Sucio':
            return 'Aspirar'
        elif location == 'A':
            return 'Derecha'
        elif location == 'B':
            return 'Derecha'
        elif location == 'C':
            return 'Derecha'
        elif location == 'D':
            return 'Izquierda'
        else:
            return 'NoOp'

class RandomSimpleReflexAgent(Agent):
  """Agente Reactivo Simple con función de agente aleatoria (Para resolver 2.14 pregunta 2)"""
  def act(self, percept):
    location,status=percept
    if status == 'Sucio':
      return 'Aspirar'
    return random.choice(['Izquierda','Derecha'])

class SimpleReflexAgentWithStates(Agent):
  """Agente Reactivo con estado (2.14 pregunta 4)"""
  def __init__(self):
    self.visited=set()

  def act(self,percept):
    location,status=percept
    self.visited.add(location)
    if status=='Sucio':
      return 'Aspirar'
    locations=['A','B','C','D']
    idx=locations.index(location)
    if idx<len(locations)-1 and locations[idx+1] not in self.visited:
      return 'Derecha'
    if idx>0 and locations[idx-1] not in self.visited:
      return 'Izquierda'
    return 'NoOp'

# ==========================================
# 3. EVALUADOR / SIMULADOR
# ==========================================
def run_simulation(agent, environment, steps=25):
    """Ejecuta la simulación separando estrictamente Agente y Entorno (Criterios 5 y 6)"""
    for _ in range(steps):
        if environment.is_clean():
            break # Termina anticipadamente si ya limpió todo

        percept = environment.get_percept()
        action = agent.act(percept)
        environment.execute_action(action)

    return environment.performance_score

def run_experiments(num_trials=10):
    """Ejecuta múltiples ensayos y compara (Criterio 9)"""
    score_random = 0
    score_reflex = 0
    score_reflex_random = 0
    score_reflex_state = 0

    for _ in range(num_trials):
        # Configuraciones iniciales aleatorias idénticas para una comparación justa
        initial_dirt = {'A': random.choice(['Limpio', 'Sucio']),
                        'B': random.choice(['Limpio', 'Sucio']),
                        'C': random.choice(['Limpio', 'Sucio']),
                        'D': random.choice(['Limpio', 'Sucio'])}
        start_loc = random.choice(['A', 'B','C','D'])

        # Prueba Agente Aleatorio
        env_rand = Environment(initial_dirt=initial_dirt, initial_agent_loc=start_loc)
        agent_rand = RandomAgent()
        score_random += run_simulation(agent_rand, env_rand)

        # Prueba Agente Reactivo Simple
        env_refl = Environment(initial_dirt=initial_dirt, initial_agent_loc=start_loc)
        agent_refl = SimpleReflexAgent()
        score_reflex += run_simulation(agent_refl, env_refl)

        # Prueba Agente Reactivo Simple Aleatorio
        env_refl_rand=Environment(initial_dirt=initial_dirt,initial_agent_loc=start_loc)
        score_reflex_random += run_simulation(RandomSimpleReflexAgent(),env_refl_rand)

        #Prueba Agente Reactivo con Estado
        env_refl_state=Environment(initial_dirt=initial_dirt,initial_agent_loc=start_loc)
        score_reflex_state += run_simulation(SimpleReflexAgentWithStates(),env_refl_state)


    print(f"--- Resultados tras {num_trials} ensayos ---")
    print(f"Puntuación Promedio Agente Aleatorio: {score_random / num_trials}")
    print(f"Puntuación Promedio Agente Reactivo: {score_reflex / num_trials}")
    print(f"Puntuación Promedio Agente Reflexivo Aleatorio: {score_reflex_random / num_trials}")
    print(f"Puntuación Promedio Agente Reflexivo con Estado: {score_reflex_state / num_trials}")
    print("\n")

"""Pregunta 2 de 2.14"""
def run_experiments_question_2(num_trials=10):
    """Ejecuta múltiples ensayos y compara (Criterio 9)"""
    score_random = 0
    score_reflex = 0
    score_reflex_random = 0
    score_reflex_state = 0

    for _ in range(num_trials):
        # Configuraciones iniciales aleatorias idénticas para una comparación justa
        initial_dirt = {'A': random.choice(['Limpio', 'Sucio']),
                        'B': random.choice(['Limpio', 'Sucio']),
                        'C': random.choice(['Limpio', 'Sucio']),
                        'D': random.choice(['Limpio', 'Sucio'])}
        start_loc = random.choice(['A', 'B','C','D'])
        # Prueba Agente Aleatorio
        env_rand = Environment(initial_dirt=initial_dirt, initial_agent_loc=start_loc)
        agent_rand = RandomAgent()
        score_random += run_simulation(agent_rand, env_rand)

        # Prueba Agente Reactivo Simple
        env_refl = Environment(initial_dirt=initial_dirt, initial_agent_loc=start_loc)
        agent_refl = SimpleReflexAgent()
        score_reflex += run_simulation(agent_refl, env_refl)

        # Prueba Agente Reactivo Simple Aleatorio
        env_refl_rand=Environment(initial_dirt=initial_dirt,initial_agent_loc=start_loc)
        score_reflex_random += run_simulation(RandomSimpleReflexAgent(),env_refl_rand)

        #Prueba Agente Reactivo con Estado
        env_refl_state=Environment(initial_dirt=initial_dirt,initial_agent_loc=start_loc)
        score_reflex_state += run_simulation(SimpleReflexAgentWithStates(),env_refl_state)


    print(f"--- Resultados tras {num_trials} ensayos ---")
    #print(f"Puntuación Promedio Agente Aleatorio: {score_random / num_trials}")
    print(f"Puntuación Promedio Agente Reactivo: {score_reflex / num_trials}")
    print(f"Puntuación Promedio Agente Reflexivo Aleatorio: {score_reflex_random / num_trials}")
    #print(f"Puntuación Promedio Agente Reflexivo con Estado: {score_reflex_state / num_trials}")
    print("\n")


"""Pregunta 3 de 2.14"""
def run_experiments_designed_environment_random_agent_perform_poorly(num_trials=10):
    """Ejecuta múltiples ensayos y compara (Criterio 9)"""
    score_random = 0
    score_reflex = 0
    score_reflex_random = 0
    score_reflex_state = 0

    for _ in range(num_trials):
        #Las configuraciones iniciales ya no son aleatorias.
        #Se escogió un entorno en donde el agente aleatorio tendrá un pesimo rendimiento.
        initial_dirt = {'A': 'Limpio',
                        'B': 'Limpio',
                        'C': 'Limpio',
                        'D': 'Sucio'}
        start_loc = 'A'

        # Prueba Agente Aleatorio
        env_rand = Environment(initial_dirt=initial_dirt, initial_agent_loc=start_loc)
        agent_rand = RandomAgent()
        score_random += run_simulation(agent_rand, env_rand)

        # Prueba Agente Reactivo Simple
        env_refl = Environment(initial_dirt=initial_dirt, initial_agent_loc=start_loc)
        agent_refl = SimpleReflexAgent()
        score_reflex += run_simulation(agent_refl, env_refl)

        # Prueba Agente Reactivo Simple Aleatorio
        env_refl_rand=Environment(initial_dirt=initial_dirt,initial_agent_loc=start_loc)
        score_reflex_random += run_simulation(RandomSimpleReflexAgent(),env_refl_rand)

        #Prueba Agente Reactivo con Estado
        env_refl_state=Environment(initial_dirt=initial_dirt,initial_agent_loc=start_loc)
        score_reflex_state += run_simulation(SimpleReflexAgentWithStates(),env_refl_state)

    print(f"--- Resultados tras {num_trials} ensayos ---")
    print(f"Puntuación Promedio Agente Aleatorio: {score_random / num_trials}")
    print(f"Puntuación Promedio Agente Reactivo: {score_reflex / num_trials}")
    print(f"Puntuación Promedio Agente Reflexivo Aleatorio: {score_reflex_random / num_trials}")
    #print(f"Puntuación Promedio Agente Reflexivo con Estado: {score_reflex_state / num_trials}")
    print("\n")

"""Pregunta 4 de 2.14"""
def run_experiments_question_4(num_trials=10):
    """Ejecuta múltiples ensayos y compara (Criterio 9)"""
    score_random = 0
    score_reflex = 0
    score_reflex_random = 0
    score_reflex_state = 0

    for _ in range(num_trials):
        # Configuraciones iniciales aleatorias idénticas para una comparación justa
        initial_dirt = {'A': random.choice(['Limpio', 'Sucio']),
                        'B': random.choice(['Limpio', 'Sucio']),
                        'C': random.choice(['Limpio', 'Sucio']),
                        'D': random.choice(['Limpio', 'Sucio'])}
        start_loc = random.choice(['A', 'B','C','D'])

        # Prueba Agente Aleatorio
        env_rand = Environment(initial_dirt=initial_dirt, initial_agent_loc=start_loc)
        agent_rand = RandomAgent()
        score_random += run_simulation(agent_rand, env_rand)

        # Prueba Agente Reactivo Simple
        env_refl = Environment(initial_dirt=initial_dirt, initial_agent_loc=start_loc)
        agent_refl = SimpleReflexAgent()
        score_reflex += run_simulation(agent_refl, env_refl)

        # Prueba Agente Reactivo Simple Aleatorio
        env_refl_rand=Environment(initial_dirt=initial_dirt,initial_agent_loc=start_loc)
        score_reflex_random += run_simulation(RandomSimpleReflexAgent(),env_refl_rand)

        #Prueba Agente Reactivo con Estado
        env_refl_state=Environment(initial_dirt=initial_dirt,initial_agent_loc=start_loc)
        score_reflex_state += run_simulation(SimpleReflexAgentWithStates(),env_refl_state)


    print(f"--- Resultados tras {num_trials} ensayos ---")
   # print(f"Puntuación Promedio Agente Aleatorio: {score_random / num_trials}")
    print(f"Puntuación Promedio Agente Reactivo: {score_reflex / num_trials}")
    #print(f"Puntuación Promedio Agente Reflexivo Aleatorio: {score_reflex_random / num_trials}")
    print(f"Puntuación Promedio Agente Reflexivo con Estado: {score_reflex_state / num_trials}")
    print("\n")


# Ejecutar el experimento
if __name__ == "__main__":
  print("Ejercicio 2.14")
  print("Pregunta 2")
  print("---Prueba de comparación entre un agente de reflejo simple y un agente de reflejo con una función de agente aleatoria")
  run_experiments_question_2(100)
  print("Pregunta 3")
  print("---Prueba de diseño donde el entorno causa que el agente aleatorio tenga un rendimiento pésimo---")
  run_experiments_designed_environment_random_agent_perform_poorly(100)
  print("Pregunta 4")
  print("---Prueba si un agente de reflejo de estado puede superar un agente de reflejo simple")
  run_experiments_question_4(100)



Ejercicio 2.14
Pregunta 2
---Prueba de comparación entre un agente de reflejo simple y un agente de reflejo con una función de agente aleatoria
--- Resultados tras 100 ensayos ---
Puntuación Promedio Agente Reactivo: 0.29
Puntuación Promedio Agente Reflexivo Aleatorio: 10.36


Pregunta 3
---Prueba de diseño donde el entorno causa que el agente aleatorio tenga un rendimiento pésimo---
--- Resultados tras 100 ensayos ---
Puntuación Promedio Agente Aleatorio: -5.78
Puntuación Promedio Agente Reactivo: 7.0
Puntuación Promedio Agente Reflexivo Aleatorio: -1.45


Pregunta 4
---Prueba si un agente de reflejo de estado puede superar un agente de reflejo simple
--- Resultados tras 100 ensayos ---
Puntuación Promedio Agente Reactivo: 1.96
Puntuación Promedio Agente Reflexivo con Estado: 13.67


